In [1]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import rand, when, pandas_udf, PandasUDFType
from pyspark.sql.types import BooleanType
import pandas as pd

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [2]:
# Create a new SparkSession
spark = (SparkSession
         .builder
         .appName("broadcast-variables")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "2g")
         .getOrCreate())

# Set log level to ERROR
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/01 12:22:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Create some sample data frames
# A large data frame with 1 million rows
large_df = (spark.range(0, 1000000)
            .withColumn("salary", 100*(rand() * 100).cast("int"))
            .withColumn("gender", when((rand() * 2).cast("int") == 0, "M").otherwise("F"))
            .withColumn("country_code", 
                        when((rand() * 4).cast("int") == 0, "US")
                        .when((rand() * 4).cast("int") == 1, "CN")
                        .when((rand() * 4).cast("int") == 2, "IN")
                        .when((rand() * 4).cast("int") == 3, "BR")))

In [4]:
large_df.show(5)

+---+------+------+------------+
| id|salary|gender|country_code|
+---+------+------+------------+
|  0|  8100|     F|        null|
|  1|   800|     M|        null|
|  2|  4800|     M|          CN|
|  3|  9700|     F|          CN|
|  4|  3400|     M|        null|
+---+------+------+------------+
only showing top 5 rows



In [5]:
lookup = {
    "US": "United States",
    "CN": "China",
    "IN": "India",
    "BR": "Brazil",
    "RU": "Russia"
}

In [6]:
broadcast_lookup = spark.sparkContext.broadcast(lookup)

In [10]:
@pandas_udf('string', PandasUDFType.SCALAR)
def country_convert(s):
    return s.map(broadcast_lookup.value)

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [9]:
large_df.withColumn(
    "country_name",
    country_convert(large_df.country_code)
)

NameError: name 'country_convert' is not defined

In [11]:
@pandas_udf(BooleanType(), PandasUDFType.SCALAR)
def filter_unknown_country(s):
    return s.isin(broadcast_lookup.value)

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [12]:
large_df.filter(filter_unknown_country(large_df.country_code)).show(5)

NameError: name 'filter_unknown_country' is not defined

In [ ]:
# spark.stop()